# Assignment 2 — Word Embeddings Comparison

**Student ID:** 2305467  
**Course:** Natural Language Processing — Fall 2026

This notebook compares locally trained Word2Vec Skip-gram and GloVe embeddings on the assigned document-classification dataset. It covers preprocessing, dimensionality reduction, semantic word-pair inspection, and downstream classification. All metrics and plots are generated from the current CSV by `scripts/run_analysis.py`; no results are hard-coded.

## Reproducibility and evaluation design

- Fixed seed: `42`; stratified 80/20 train/test split.
- Vocabulary and embedding models are trained on training documents only.
- Both models use 50-dimensional vectors and a context window of five; Word2Vec uses Skip-gram (`sg=1`).
- Document vectors are the mean of known word vectors. A class-balanced Logistic Regression classifier is evaluated on the same held-out split.
- GloVe is implemented locally as a compact educational weighted least-squares model over co-occurrence counts. It is not a pretrained large-scale GloVe release.
- PCA and t-SNE are exploratory projections. Curated word pairs are illustrative, not a gold-standard semantic benchmark; coverage is reported.


## 1. Dataset inspection and preprocessing

The loader detects common text/label column names and prints the actual selected columns. It removes blank text/labels and duplicate text-label rows, applies lowercase alphabetic tokenization, removes English stop words, and uses Porter stemming. We inspect the real schema and class counts before interpreting results.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

DATA_PATH = Path('df_file.csv')
OUTPUT_DIR = Path('report/generated')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH.resolve()}')

from scripts.run_analysis import load_dataset, preprocess
raw_df, text_column, label_column = load_dataset(DATA_PATH)
print(f'Raw dataset shape: {raw_df.shape}')
print(f'Text column: {text_column!r}; label column: {label_column!r}')
display(raw_df[[text_column, label_column]].head())
display(raw_df[label_column].value_counts(dropna=False).rename_axis('label').to_frame('documents'))


## 2. Train embeddings and evaluate

The following cell executes the same pipeline used by the command-line workflow. It writes metrics to JSON/CSV and figures to `report/generated/figures/`.


In [ ]:
command = [sys.executable, 'scripts/run_analysis.py', '--data', str(DATA_PATH), '--output', str(OUTPUT_DIR)]
completed = subprocess.run(command, check=True, text=True, capture_output=True)
print(completed.stdout)
metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
summary = pd.DataFrame([
    {key: item[key] for key in ('model', 'vocabulary_size', 'train_accuracy', 'test_accuracy', 'macro_f1', 'weighted_f1', 'test_oov_document_rate')}
    for item in metrics['results']
])
display(summary.style.format({
    'train_accuracy': '{:.3f}', 'test_accuracy': '{:.3f}',
    'macro_f1': '{:.3f}', 'weighted_f1': '{:.3f}', 'test_oov_document_rate': '{:.1%}'
}))


## 3. Word-pair semantic similarity

Cosine similarity is reported for curated domain-related word pairs when both words are present after the same preprocessing pipeline. Low coverage is a limitation, not evidence that omitted pairs have low similarity. The list is illustrative and not a human-annotated benchmark.


In [ ]:
## 3. Semantic similarity and topic coherence

Cosine similarity is reported for curated domain-related word pairs when both words are present after the same preprocessing pipeline. Low coverage is a limitation, not evidence that omitted pairs have low similarity. The list is illustrative and not a gold-standard semantic benchmark.

Topic coherence is approximated by mean pairwise cosine similarity among frequent class-associated terms selected using training data only. This is an exploratory class-conditioned cohesion score, **not** a standard NPMI/topic-model coherence benchmark; class labels inform term selection, so it should not be treated as an independent unsupervised test.


for item in metrics['results']:
    print(f"\n{item['model']} — word-pair coverage: {item['semantic_similarity']['evaluated_pairs']}/{item['semantic_similarity']['candidate_pairs']} ({item['semantic_similarity']['coverage']:.1%})")
    display(pd.DataFrame(item['semantic_similarity']['pairs']))
    coherence = item['topic_coherence']
    print('Training-only class-conditioned topic-coherence proxy:', coherence['overall_mean_pairwise_cosine'])
    display(pd.DataFrame.from_dict(coherence['by_class'], orient='index'))


In [ ]:
for item in metrics['results']:
    print(f"\n{item['model']}")
    display(pd.DataFrame(item['per_class']).T)
    print('Confusion matrix (rows=true, columns=predicted):')
    display(pd.DataFrame(item['confusion_matrix'], index=metrics['dataset']['labels'], columns=metrics['dataset']['labels']))
    safe_name = 'word2vec' if item['model'].startswith('Word2Vec') else 'glove'
    display(Image(filename=str(OUTPUT_DIR / 'figures' / f'{safe_name}_confusion.png')))


## 5. PCA and t-SNE visualizations

PCA is a linear projection that preserves directions of high variance; t-SNE emphasizes local neighborhood structure and may distort global distances. Cluster appearance can depend on vocabulary selection, model settings, and random initialization. These plots are descriptive and are not by themselves a semantic evaluation.


In [ ]:
## 6. Findings and limitations

Use the computed results below to write the final interpretation. Do not assume either model wins before reviewing the held-out metrics, per-class metrics, OOV coverage, semantic-pair coverage, and the exploratory class-conditioned cohesion score. The cohesion score is not a standard topic-coherence benchmark and uses training labels to select terms.


## 6. Findings and limitations

Use the computed results below to write the final interpretation. Do not assume either model wins before reviewing the held-out metrics and semantic-pair coverage. This dataset is small and domain-specific; the local GloVe implementation and curated word pairs are educational approximations, and results may differ from pretrained embeddings.


In [ ]:
best = max(metrics['results'], key=lambda item: item['test_accuracy'])
print(f"Dataset: {metrics['dataset']['clean_rows']} cleaned documents; {metrics['dataset']['train_size']} train / {metrics['dataset']['test_size']} test.")
print('Class counts:', metrics['dataset']['class_counts'])
print('Selected best test accuracy (ties follow listed order):', best['model'], f"{best['test_accuracy']:.3f}")
print('Interpret the accuracy alongside macro F1, per-class metrics, OOV document rate, and semantic-pair coverage.')
display(pd.DataFrame(metrics['results']).drop(columns=['per_class', 'confusion_matrix', 'semantic_similarity']))
